# 02 — Modeling

Order of work:
1. Walk-forward CV on the **training** period to compare models and tune hyperparameters.
2. Check the chosen setup once on **validation** (2021–2022).
3. Freeze everything, then run the **test** period (2023+) once, and backtest it.

Metrics: macro F1 and balanced accuracy (plain accuracy is misleading with imbalanced classes). ~0.35–0.40 is normal for three classes; anything far higher means a bug or a leak.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spysignal import data, features, labels, splits
from spysignal.features import FEATURES

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "lines.linewidth": 1.5,
})
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#a3a29d"
CLASS_COLORS = {"Sell": "#e34948", "Hold": GRAY, "Buy": BLUE}   # diverging: red - gray - blue
NAMES = {-1: "Sell", 0: "Hold", 1: "Buy"}

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

In [2]:
# data/raw.csv is created once with data.download(); everyone uses the same file.
if not data.RAW_FILE.exists():
    data.download()
df = data.load()

X_all = features.build_features(df)
fwd = labels.forward_return(df)
scale = labels.threshold_scale(df)
usable = X_all[list(FEATURES)].notna().all(axis=1) & fwd.notna() & scale.notna()
masks = splits.split_masks(df.index, usable)

k = labels.fit_k(fwd[masks["train"]], scale[masks["train"]])   # fitted on TRAIN only
y = labels.make_labels(fwd, scale, k)
print(f"k = {k:.3f}  (Buy if 5-day return > {k:.2f} x expected 5-day volatility)")
for name, m in masks.items():
    print(f"{name:5s}: {m.sum():5d} rows  {df.index[m].min().date()} -> {df.index[m].max().date()}")

k = 0.379  (Buy if 5-day return > 0.38 x expected 5-day volatility)
train:  3636 rows  2006-07-17 -> 2020-12-22
val  :   497 rows  2021-01-04 -> 2022-12-21
test :   940 rows  2023-01-03 -> 2026-10-01


In [3]:
cols = list(FEATURES)
Xtr, ytr = X_all.loc[masks["train"], cols], y[masks["train"]].astype(int)
Xva, yva = X_all.loc[masks["val"], cols], y[masks["val"]].astype(int)

## 1. Walk-forward CV on the training period
Each model is a `Pipeline`, so the scaler is re-fit inside every fold (fitting it on all data would leak future means/variances). `TimeSeriesSplit(gap=6)` leaves a 6-day gap so a training label never overlaps the fold it is scored on. Class imbalance: `class_weight='balanced'` (sample weights for XGBoost).

In [4]:
models = {
    "dummy (majority)": DummyClassifier(strategy="most_frequent"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)),
    "random forest": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=50,
                                            class_weight="balanced", n_jobs=-1, random_state=0),
    "xgboost": XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=0),
}

def fit(model, X, y_):
    if isinstance(model, XGBClassifier):            # XGBoost needs labels 0..2 and sample weights
        return model.fit(X, y_ + 1, sample_weight=compute_sample_weight("balanced", y_))
    return model.fit(X, y_)

def predict(model, X):
    p = model.predict(X)
    return p - 1 if isinstance(model, XGBClassifier) else p

rows = []
for name, model in models.items():
    for fold, (tr, te) in enumerate(splits.cv_splitter().split(Xtr)):
        fit(model, Xtr.iloc[tr], ytr.iloc[tr])
        pred = predict(model, Xtr.iloc[te])
        rows.append({"model": name, "fold": fold,
                     "macro_f1": f1_score(ytr.iloc[te], pred, average="macro"),
                     "balanced_acc": balanced_accuracy_score(ytr.iloc[te], pred)})
cv = pd.DataFrame(rows)
display(cv.groupby("model")[["macro_f1", "balanced_acc"]].agg(["mean", "std"]).round(3))

macro_f1        balanced_acc       
                     mean    std         mean    std
model                                               
dummy (majority)    0.199  0.007        0.333  0.000
logistic            0.367  0.021        0.386  0.026
random forest       0.372  0.022        0.393  0.022
xgboost             0.360  0.038        0.383  0.016

## 2. Sanity check: shuffled labels
Train on randomly shuffled labels. Scores must drop to chance (~0.33). If they don't, something in the pipeline is leaking.

In [5]:
rng = np.random.default_rng(0)
shuffled = pd.Series(rng.permutation(ytr.to_numpy()), index=ytr.index)
m = models["logistic"]
scores = [balanced_accuracy_score(shuffled.iloc[te], fit(m, Xtr.iloc[tr], shuffled.iloc[tr]).predict(Xtr.iloc[te]))
          for tr, te in splits.cv_splitter().split(Xtr)]
print(f"balanced accuracy on shuffled labels: {np.mean(scores):.3f}")

balanced accuracy on shuffled labels: 0.321


## 3. Hyperparameter tuning
TODO: small grid per model, scored by mean CV macro F1 on the training period only. Keep grids small: every extra setting tried is another chance to overfit to noise.

## 4. Validation check (2021–2022)
TODO: fit the chosen model on all training rows, predict validation once, report confusion matrix and macro F1. Also split it into two questions: *Hold vs move* and *Buy vs Sell*.

## 5. Final test + backtest (run once)
TODO, only after sections 3–4 are frozen: refit on train + validation, predict 2023+, then `backtest.run(signals, df['open'])` vs `backtest.buy_and_hold(df['open'])` with the 0.1% fee. Plot both portfolio values.

## 6. Lessons learned
TODO.